# Validate Synchronization (RQ1)

This notebook calculates the intra-individual Pearson correlation coefficient ($r$) across the ActiGraph, EmotiBit, Bangle.js, and Polar HR devices.

Because physical exertion generates different baseline acceleration forces for different individuals, computing a global correlation across all stitched data introduces inter-individual variance (Simpson's Paradox). To correctly evaluate the success of the temporal alignment pipeline, we calculate the correlation for each participant individually, and then average the results across the cohort.

The Polar HR stream is included as a separate device to validate that the cardiovascular response captured by the chest strap is temporally aligned with the kinetic movement signals from the accelerometers.

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the aligned 5-second overlapping epochs
df = pd.read_csv(r'Acc_pipe/data/processed/master_epochs.csv')

# Ensure all devices have their required data for the epoch
df = df.dropna(subset=['vm_mean_actigraph', 'vm_mean_emotibit', 'vm_mean_bangle', 'hr_polar'])

correlations = []

# Calculate intra-individual correlation
for subject, group in df.groupby('subject_id'):
    # Require at least 10 valid epochs and non-zero variance to calculate Pearson r
    if len(group) > 10 and group['vm_mean_actigraph'].std() > 0 and group['vm_mean_emotibit'].std() > 0 and group['vm_mean_bangle'].std() > 0 and group['hr_polar'].std() > 0:
        r_act_emo = group['vm_mean_actigraph'].corr(group['vm_mean_emotibit'])
        r_act_ban = group['vm_mean_actigraph'].corr(group['vm_mean_bangle'])
        r_emo_ban = group['vm_mean_emotibit'].corr(group['vm_mean_bangle'])
        r_act_pol = group['vm_mean_actigraph'].corr(group['hr_polar'])
        r_emo_pol = group['vm_mean_emotibit'].corr(group['hr_polar'])
        r_ban_pol = group['vm_mean_bangle'].corr(group['hr_polar'])
        
        correlations.append({
            'Subject': subject,
            'Actigraph_EmotiBit': r_act_emo,
            'Actigraph_Bangle': r_act_ban,
            'EmotiBit_Bangle': r_emo_ban,
            'Actigraph_Polar': r_act_pol,
            'EmotiBit_Polar': r_emo_pol,
            'Bangle_Polar': r_ban_pol
        })

# Convert to DataFrame
corr_df = pd.DataFrame(correlations)

# Exclude the 6 participants with known data quality issues (e.g. sensor fell off, failed to record)
# This exactly mirrors the 'Filter Bad Data' toggle logic in the Analytics Dashboard
bad_subjects = [2004, 2005, 2008, 2014, 2019, 2032]
corr_df = corr_df[~corr_df['Subject'].isin(bad_subjects)]

# Calculate the cohort average
avg_correlations = corr_df.mean(numeric_only=True)

print("=== Average Intra-Individual Correlation ===")
print(f"Actigraph vs EmotiBit: {avg_correlations['Actigraph_EmotiBit']:.2f}")
print(f"Actigraph vs Bangle.js: {avg_correlations['Actigraph_Bangle']:.2f}")
print(f"EmotiBit vs Bangle.js: {avg_correlations['EmotiBit_Bangle']:.2f}")
print(f"Actigraph vs Polar HR: {avg_correlations['Actigraph_Polar']:.2f}")
print(f"EmotiBit vs Polar HR: {avg_correlations['EmotiBit_Polar']:.2f}")
print(f"Bangle.js vs Polar HR: {avg_correlations['Bangle_Polar']:.2f}")

=== Average Intra-Individual Correlation ===
Actigraph vs EmotiBit: 0.95
Actigraph vs Bangle.js: 0.93
EmotiBit vs Bangle.js: 0.93
Actigraph vs Polar HR: 0.96
EmotiBit vs Polar HR: 0.93
Bangle.js vs Polar HR: 0.93


### Generate Thesis Table
The code below formats the mathematical output into a 4x4 correlation matrix suitable for inclusion in the thesis text.

In [6]:
matrix = pd.DataFrame({
    'ActiGraph': [1.00, avg_correlations['Actigraph_EmotiBit'], avg_correlations['Actigraph_Bangle'], avg_correlations['Actigraph_Polar']],
    'EmotiBit': [avg_correlations['Actigraph_EmotiBit'], 1.00, avg_correlations['EmotiBit_Bangle'], avg_correlations['EmotiBit_Polar']],
    'Bangle.js': [avg_correlations['Actigraph_Bangle'], avg_correlations['EmotiBit_Bangle'], 1.00, avg_correlations['Bangle_Polar']],
    'Polar HR': [avg_correlations['Actigraph_Polar'], avg_correlations['EmotiBit_Polar'], avg_correlations['Bangle_Polar'], 1.00]
}, index=['ActiGraph', 'EmotiBit', 'Bangle.js', 'Polar HR'])

print("\nCorrelation Matrix (as seen in Thesis Table 5.1):\n")
print(matrix.round(2))


Correlation Matrix (as seen in Thesis Table 5.1):

           ActiGraph  EmotiBit  Bangle.js  Polar HR
ActiGraph       1.00      0.95       0.93      0.96
EmotiBit        0.95      1.00       0.93      0.93
Bangle.js       0.93      0.93       1.00      0.93
Polar HR        0.96      0.93       0.93      1.00
